
# Revision 04 — Held-out top-m MS-AA cross-fitting

This notebook is built directly from the data structures and reconstruction code used in:

- `001_fit_msaa_flexible.ipynb`
- `002_decode_msaa.ipynb`

It addresses the reviewer concern that the original top-m analysis used the same participants to rank individual archetypes and evaluate the selected top-m reconstruction.

For each condition, model order K, and repetition:

1. Split the condition's participants into two outer halves A and B.
2. Use only A to rank individual archetypes.
3. Select top-m from that A-derived ranking.
4. Evaluate those selected archetypes using only B.
5. Reverse the roles of A and B.

Within each outer half, timepoint matching is performed exactly as in `002_decode_msaa.ipynb`: participants are repeatedly divided into two folds, reconstructed data are averaged within fold, and timepoints are matched by correlation.

Thus, no participant used to rank the archetypes contributes to the held-out top-m evaluation score.

The MS-AA basis itself remains the already-fitted shared across-condition basis. This notebook specifically tests whether the ranking/selection result generalizes across participants, which is the circularity issue raised by the reviewer.


In [ ]:

from pathlib import Path

ANALYSIS_TYPE = "spatial"   # "spatial" or "temporal"
FIT_SCOPE = "across"

FIT_LOAD_DIR = "msaa_flexible_outputs_npz"
OUTPUT_DIR = Path(f"msaa_heldout_topm_outputs_{ANALYSIS_TYPE}_{FIT_SCOPE}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CONDITIONS = ["intact", "word", "rest"]
CONDITIONS_TO_RUN = CONDITIONS
# CONDITIONS_TO_RUN = ["intact"]

K_VALUES = None
QUICK_K_VALUES = [300, 500, 700]
# QUICK_K_VALUES = [35, 48, 70, 88]

TOP_M_VALUES = [1, 2, 3, 5, 10]

N_OUTER_REPS = 20
NFOLDS_DECODE = 2
NREPS_DECODE = 20
RNG_SEED = 42

print("ANALYSIS_TYPE:", ANALYSIS_TYPE)
print("FIT_SCOPE:", FIT_SCOPE)
print("FIT_LOAD_DIR:", FIT_LOAD_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR.resolve())


In [ ]:

%matplotlib inline

import os
import re
import numpy as np
import pandas as pd
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
from IPython.display import display


## 1. Load saved fits exactly as `002_decode_msaa.ipynb` does

In [ ]:

def load_msaa_npz(path):
    data = np.load(path, allow_pickle=True)

    results_subj = data["results_subj"].tolist()
    if isinstance(results_subj, np.ndarray):
        results_subj = results_subj.tolist()

    out = {
        "K": int(data["K"]) if "K" in data else None,
        "results_subj": results_subj,
    }

    for key in [
        "condition_labels_str",
        "condition_codes",
        "condition_names",
        "C",
        "cost_fun",
        "varexpl",
        "secs",
        "analysis_type",
        "fit_scope",
    ]:
        if key in data:
            val = data[key]
            try:
                out[key] = val.tolist()
            except Exception:
                out[key] = val

    return out


def fit_path_for_config(analysis_type, fit_scope, K, condition=None):
    if fit_scope == "across":
        return os.path.join(
            FIT_LOAD_DIR,
            f"{analysis_type}AA_across_acrossCond_K{K}.npz"
        )
    elif fit_scope == "within":
        if condition is None:
            raise ValueError("condition is required for within-condition fits")
        return os.path.join(
            FIT_LOAD_DIR,
            f"{analysis_type}AA_within_{condition}_K{K}.npz"
        )
    else:
        raise ValueError("fit_scope must be 'within' or 'across'")


def load_fit(analysis_type, fit_scope, K, condition=None):
    path = fit_path_for_config(
        analysis_type, fit_scope, K, condition=condition
    )
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing fit file: {path}")

    fit = load_msaa_npz(path)
    fit["fit_source_file"] = path
    fit["fit_source_basename"] = os.path.basename(path)
    return fit


def get_condition_subjects_from_across_fit(fit, condition):
    labels = np.asarray(fit["condition_labels_str"])
    idx = np.where(labels == condition)[0]
    return [fit["results_subj"][i] for i in idx]


def get_subjects_for_condition(analysis_type, fit_scope, K, condition):
    if fit_scope == "across":
        fit = load_fit(analysis_type, fit_scope, K)
        return get_condition_subjects_from_across_fit(fit, condition), fit
    else:
        fit = load_fit(
            analysis_type, fit_scope, K, condition=condition
        )
        return fit["results_subj"], fit


In [ ]:

def discover_k_values(analysis_type, fit_scope="across"):
    if fit_scope != "across":
        raise NotImplementedError("Auto-discovery here is configured for across-condition fits.")

    pat = re.compile(
        rf"^{analysis_type}AA_across_acrossCond_K(\d+)\.npz$"
    )

    ks = []
    for f in Path(FIT_LOAD_DIR).glob("*.npz"):
        m = pat.match(f.name)
        if m:
            ks.append(int(m.group(1)))

    return sorted(set(ks))


available_K = discover_k_values(ANALYSIS_TYPE, FIT_SCOPE)
print("Available K values:", available_K)

if K_VALUES is None:
    K_VALUES = available_K

if QUICK_K_VALUES is not None:
    K_VALUES = [k for k in QUICK_K_VALUES if k in available_K]

print("K values to run:", K_VALUES)

if len(K_VALUES):
    fit0 = load_fit(ANALYSIS_TYPE, FIT_SCOPE, K_VALUES[0])
    print("\nFirst fit:", fit0["fit_source_basename"])
    print("N results_subj:", len(fit0["results_subj"]))
    print("Condition counts:")
    display(pd.Series(fit0["condition_labels_str"]).value_counts())
    print("subj0 sXC:", np.asarray(fit0["results_subj"][0]["sXC"]).shape)
    print("subj0 S:", np.asarray(fit0["results_subj"][0]["S"]).shape)


## 2. Reconstruction and decoding — same implementation as `002_decode_msaa.ipynb`

In [ ]:

def to_float_array(x):
    return np.asarray(x, dtype=float)


def reconstruct_from_archetypes(sub, analysis_type, archetype_indices=None):
    sXC = to_float_array(sub["sXC"])
    S = to_float_array(sub["S"])

    if archetype_indices is None:
        idx = np.arange(S.shape[0])
    else:
        idx = np.asarray(list(archetype_indices), dtype=int)

    if analysis_type == "spatial":
        Xhat = sXC[:, idx] @ S[idx, :]
    elif analysis_type == "temporal":
        Xhat = (sXC[:, idx] @ S[idx, :]).T
    else:
        raise ValueError("analysis_type must be 'spatial' or 'temporal'")

    Xhat = (
        Xhat - Xhat.mean(axis=0, keepdims=True)
    ) / (
        Xhat.std(axis=0, keepdims=True) + 1e-8
    )

    return Xhat


def build_recon_stack(subjects, analysis_type, archetype_indices=None):
    return np.stack([
        reconstruct_from_archetypes(
            sub,
            analysis_type,
            archetype_indices=archetype_indices
        )
        for sub in subjects
    ], axis=0)


def decoder(corrs):
    out = pd.DataFrame({
        "rank": [0.0],
        "accuracy": [0.0],
        "error": [0.0],
    })

    T = corrs.shape[0]

    for t in range(T):
        decoded_ind = int(np.argmax(corrs[t, :]))

        out.loc[0, "error"] += np.mean(np.abs(decoded_ind - t)) / T
        out.loc[0, "accuracy"] += (decoded_ind == t)
        out.loc[0, "rank"] += np.mean(
            (corrs[t, :] <= corrs[t, t]).astype(int)
        )

    out["error"] /= T
    out["accuracy"] /= T
    out["rank"] /= T

    return out


def get_xval_assignments(ndata, nfolds, rng=None):
    rng = np.random.default_rng() if rng is None else rng

    group_assignments = np.zeros(ndata, dtype=int)
    groupsize = int(np.ceil(ndata / nfolds))

    for i in range(1, nfolds):
        inds = np.arange(
            i * groupsize,
            min((i + 1) * groupsize, ndata)
        )
        group_assignments[inds] = i

    rng.shuffle(group_assignments)
    return group_assignments


def run_timepoint_decoding(
    recon_stack,
    nfolds=2,
    nreps=20,
    seed=42,
):
    N, T, V = recon_stack.shape

    rng = np.random.default_rng(seed)
    rows = []

    for rep in range(nreps):
        fold_ids = get_xval_assignments(
            N,
            nfolds,
            rng=rng
        )

        for fold in range(nfolds):
            in_mask = (fold_ids == fold)
            out_mask = ~in_mask

            in_mean = recon_stack[in_mask].mean(axis=0)
            out_mean = recon_stack[out_mask].mean(axis=0)

            corrs = 1.0 - cdist(
                in_mean,
                out_mean,
                metric="correlation"
            )

            corrs = np.nan_to_num(
                corrs,
                nan=0.0,
                posinf=0.0,
                neginf=0.0
            )

            res = decoder(corrs)
            res["rep"] = rep
            res["fold"] = fold
            rows.append(res)

    return pd.concat(rows, ignore_index=True)


def summarize_decoding_runs(dec_df):
    return {
        "mean": float(dec_df["accuracy"].mean()),
        "err": float(
            dec_df["accuracy"].std()
            / np.sqrt(max(len(dec_df), 1))
        ),
        "std": float(dec_df["accuracy"].std()),
        "count": int(len(dec_df)),
        "rank_mean": float(dec_df["rank"].mean()),
        "error_mean": float(dec_df["error"].mean()),
    }


def decode_subjects(
    subjects,
    analysis_type,
    archetype_indices=None,
    nfolds=2,
    nreps=20,
    seed=42,
):
    recon_stack = build_recon_stack(
        subjects,
        analysis_type,
        archetype_indices=archetype_indices
    )

    dec_df = run_timepoint_decoding(
        recon_stack,
        nfolds=nfolds,
        nreps=nreps,
        seed=seed
    )

    return dec_df, summarize_decoding_runs(dec_df)



## 3. Cross-fitted ranking and evaluation

`ranking_subjects` and `evaluation_subjects` are non-overlapping.

Within the ranking half, each archetype is decoded independently and mean accuracy across the same repeated two-fold procedure is used to rank archetypes. The selected top-m subset is then evaluated on the held-out participant half.


In [ ]:

def split_outer_subject_indices(n_subjects, rng):
    idx = np.arange(n_subjects)
    rng.shuffle(idx)

    midpoint = n_subjects // 2
    A = idx[:midpoint]
    B = idx[midpoint:]

    return A, B


def subset_subjects(subjects, indices):
    return [subjects[int(i)] for i in indices]


def get_n_archetypes(subjects):
    return int(np.asarray(subjects[0]["S"]).shape[0])


def rank_archetypes_on_subjects(
    ranking_subjects,
    analysis_type,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=42,
):
    K = get_n_archetypes(ranking_subjects)
    rows = []

    for archetype in range(K):
        _, summary = decode_subjects(
            ranking_subjects,
            analysis_type,
            archetype_indices=[archetype],
            nfolds=nfolds,
            nreps=nreps,
            seed=seed,
        )

        rows.append({
            "archetype": int(archetype),
            "mean_accuracy": summary["mean"],
            "sem_accuracy": summary["err"],
            "rank_mean": summary["rank_mean"],
            "error_mean": summary["error_mean"],
        })

    per_arch = pd.DataFrame(rows)

    ranked = (
        per_arch
        .sort_values(
            ["mean_accuracy", "rank_mean", "archetype"],
            ascending=[False, False, True]
        )
        ["archetype"]
        .astype(int)
        .tolist()
    )

    return ranked, per_arch


def evaluate_topm_on_subjects(
    evaluation_subjects,
    analysis_type,
    ranked_archetypes,
    top_m_values=TOP_M_VALUES,
    nfolds=NFOLDS_DECODE,
    nreps=NREPS_DECODE,
    seed=42,
):
    K = get_n_archetypes(evaluation_subjects)

    _, full_summary = decode_subjects(
        evaluation_subjects,
        analysis_type,
        archetype_indices=None,
        nfolds=nfolds,
        nreps=nreps,
        seed=seed,
    )

    example = reconstruct_from_archetypes(
        evaluation_subjects[0],
        analysis_type,
        archetype_indices=None
    )
    T = example.shape[0]
    chance = 1.0 / T

    rows = []

    for top_m in top_m_values:
        m = min(int(top_m), K)
        selected = ranked_archetypes[:m]

        _, summary = decode_subjects(
            evaluation_subjects,
            analysis_type,
            archetype_indices=selected,
            nfolds=nfolds,
            nreps=nreps,
            seed=seed,
        )

        denom = full_summary["mean"] - chance

        retention = (
            (summary["mean"] - chance) / denom
            if np.abs(denom) > 1e-12
            else np.nan
        )

        rows.append({
            "top_m": m,
            "selected_archetypes": str(selected),
            "mean": summary["mean"],
            "err": summary["err"],
            "std": summary["std"],
            "count": summary["count"],
            "rank_mean": summary["rank_mean"],
            "error_mean": summary["error_mean"],
            "full_mean_same_eval": full_summary["mean"],
            "full_err_same_eval": full_summary["err"],
            "chance": chance,
            "retention_above_chance": retention,
        })

    return pd.DataFrame(rows)



## 4. Single-configuration smoke test

Run this before the full analysis. With the primary data it should show 36 subjects in the condition and an 18/18 outer split.


In [ ]:

# TEST_CONDITION = "intact"
# TEST_K = K_VALUES[0] if len(K_VALUES) else None

# if TEST_K is not None:
#     subjects, fit = get_subjects_for_condition(
#         ANALYSIS_TYPE,
#         FIT_SCOPE,
#         TEST_K,
#         TEST_CONDITION,
#     )

#     print("Fit:", fit["fit_source_basename"])
#     print("Condition:", TEST_CONDITION)
#     print("N subjects:", len(subjects))
#     print("K:", get_n_archetypes(subjects))

#     rng = np.random.default_rng(RNG_SEED)
#     A_idx, B_idx = split_outer_subject_indices(
#         len(subjects),
#         rng
#     )

#     A_subjects = subset_subjects(subjects, A_idx)
#     B_subjects = subset_subjects(subjects, B_idx)

#     print("Outer A:", len(A_subjects))
#     print("Outer B:", len(B_subjects))

#     ranked_A, per_arch_A = rank_archetypes_on_subjects(
#         A_subjects,
#         ANALYSIS_TYPE,
#         nfolds=NFOLDS_DECODE,
#         nreps=NREPS_DECODE,
#         seed=RNG_SEED + 100,
#     )

#     print("Top 10 archetypes learned on A:", ranked_A[:10])
#     display(
#         per_arch_A
#         .sort_values("mean_accuracy", ascending=False)
#         .head(10)
#     )

#     heldout_B = evaluate_topm_on_subjects(
#         B_subjects,
#         ANALYSIS_TYPE,
#         ranked_A,
#         top_m_values=TOP_M_VALUES,
#         nfolds=NFOLDS_DECODE,
#         nreps=NREPS_DECODE,
#         seed=RNG_SEED + 200,
#     )

#     display(heldout_B)


## 5. Full cross-fitted analysis

In [ ]:
RUN_CSV = OUTPUT_DIR / "heldout_topm_crossfit_runs.csv"
RANK_CSV = OUTPUT_DIR / "heldout_per_archetype_ranking_runs.csv"

MAX_K = 700

# -------------------------------------------------------------------------
# Load existing checkpoints
# -------------------------------------------------------------------------

if RUN_CSV.exists():
    run_df_existing = pd.read_csv(RUN_CSV)
    print(f"Loaded existing held-out results: {len(run_df_existing):,} rows")
else:
    run_df_existing = pd.DataFrame()
    print("No existing held-out run checkpoint found.")

if RANK_CSV.exists():
    rank_df_existing = pd.read_csv(RANK_CSV)
    print(f"Loaded existing ranking results: {len(rank_df_existing):,} rows")
else:
    rank_df_existing = pd.DataFrame()
    print("No existing ranking checkpoint found.")


# Keep accumulated results as DataFrames rather than starting from scratch
run_parts = []
rank_parts = []

if not run_df_existing.empty:
    run_parts.append(run_df_existing)

if not rank_df_existing.empty:
    rank_parts.append(rank_df_existing)


# -------------------------------------------------------------------------
# Helper: determine which outer reps are already complete
# -------------------------------------------------------------------------

def completed_outer_reps(df, analysis_type, fit_scope, condition, K):
    """
    An outer rep counts as complete only if BOTH directions
    (A_to_B and B_to_A) are present.
    """
    if df.empty:
        return set()

    required_cols = {
        "analysis_type",
        "fit_scope",
        "condition",
        "K",
        "outer_rep",
        "direction",
    }

    if not required_cols.issubset(df.columns):
        return set()

    sub = df[
        (df["analysis_type"] == analysis_type)
        & (df["fit_scope"] == fit_scope)
        & (df["condition"] == condition)
        & (df["K"].astype(int) == int(K))
    ]

    complete = set()

    for outer_rep, grp in sub.groupby("outer_rep"):
        directions = set(grp["direction"].dropna().astype(str))
        if {"A_to_B", "B_to_A"}.issubset(directions):
            complete.add(int(outer_rep))

    return complete


def save_checkpoints(run_parts, rank_parts):
    """
    Concatenate, remove accidental duplicate rows, and save.
    """
    if run_parts:
        run_df = pd.concat(run_parts, ignore_index=True)

        # Defensive deduplication in case a rep was ever rerun
        dedup_cols = [
            c for c in [
                "analysis_type",
                "fit_scope",
                "condition",
                "K",
                "outer_rep",
                "direction",
                "top_m",
                "decode_rep",
                "fold",
            ]
            if c in run_df.columns
        ]

        if dedup_cols:
            run_df = run_df.drop_duplicates(
                subset=dedup_cols,
                keep="last",
            )

        run_df.to_csv(RUN_CSV, index=False)

    else:
        run_df = pd.DataFrame()

    if rank_parts:
        rank_df = pd.concat(rank_parts, ignore_index=True)

        dedup_cols = [
            c for c in [
                "analysis_type",
                "fit_scope",
                "condition",
                "K",
                "outer_rep",
                "direction",
                "archetype",
                "decode_rep",
                "fold",
            ]
            if c in rank_df.columns
        ]

        if dedup_cols:
            rank_df = rank_df.drop_duplicates(
                subset=dedup_cols,
                keep="last",
            )

        rank_df.to_csv(RANK_CSV, index=False)

    else:
        rank_df = pd.DataFrame()

    return run_df, rank_df


# -------------------------------------------------------------------------
# Restrict this run to K <= 50
# -------------------------------------------------------------------------

K_VALUES_TO_RUN = [
    int(K)
    for K in K_VALUES
    if int(K) <= MAX_K
]

print("\nK values considered in this run:")
print(K_VALUES_TO_RUN)


# -------------------------------------------------------------------------
# Main loop
# -------------------------------------------------------------------------

for K in K_VALUES_TO_RUN:

    print("\n" + "=" * 90)
    print(f"K = {K}")
    print("=" * 90)

    for condition in CONDITIONS_TO_RUN:

        # Reconstruct current saved state from accumulated pieces
        current_run_df = (
            pd.concat(run_parts, ignore_index=True)
            if run_parts
            else pd.DataFrame()
        )

        current_rank_df = (
            pd.concat(rank_parts, ignore_index=True)
            if rank_parts
            else pd.DataFrame()
        )

        # Require completion in BOTH output files
        run_complete = completed_outer_reps(
            current_run_df,
            ANALYSIS_TYPE,
            FIT_SCOPE,
            condition,
            K,
        )

        rank_complete = completed_outer_reps(
            current_rank_df,
            ANALYSIS_TYPE,
            FIT_SCOPE,
            condition,
            K,
        )

        completed_reps = run_complete.intersection(rank_complete)

        # Entire condition already finished
        if len(completed_reps) >= N_OUTER_REPS:
            print(
                f"\n{ANALYSIS_TYPE} | {FIT_SCOPE} | "
                f"{condition} | K={K} "
                f"--> already complete ({N_OUTER_REPS}/{N_OUTER_REPS}); skipping"
            )
            continue

        subjects, fit = get_subjects_for_condition(
            ANALYSIS_TYPE,
            FIT_SCOPE,
            K,
            condition,
        )

        print(
            f"\n{ANALYSIS_TYPE} | {FIT_SCOPE} | "
            f"{condition} | K={K} | N={len(subjects)}"
        )

        if completed_reps:
            print(
                f"  found {len(completed_reps)}/{N_OUTER_REPS} "
                f"completed outer reps; resuming missing reps"
            )

        for outer_rep in range(N_OUTER_REPS):

            # Skip outer reps already present in BOTH checkpoint files
            if outer_rep in completed_reps:
                continue

            # -------------------------------------------------------------
            # Deterministic outer seed
            #
            # This no longer depends on how many earlier K values were
            # skipped, so restarting the cell gives the same split.
            # -------------------------------------------------------------

            condition_index = CONDITIONS_TO_RUN.index(condition)

            seed_sequence = np.random.SeedSequence([
                int(RNG_SEED),
                int(K),
                int(condition_index),
                int(outer_rep),
            ])

            outer_seed = int(
                seed_sequence.generate_state(
                    1,
                    dtype=np.uint32
                )[0]
            )

            outer_rng = np.random.default_rng(outer_seed)

            A_idx, B_idx = split_outer_subject_indices(
                len(subjects),
                outer_rng
            )

            directions = [
                ("A_to_B", A_idx, B_idx),
                ("B_to_A", B_idx, A_idx),
            ]

            outer_run_parts = []
            outer_rank_parts = []

            for direction, ranking_idx, evaluation_idx in directions:

                ranking_subjects = subset_subjects(
                    subjects,
                    ranking_idx
                )

                evaluation_subjects = subset_subjects(
                    subjects,
                    evaluation_idx
                )

                ranking_seed = (
                    RNG_SEED
                    + 100000 * outer_rep
                    + 1000 * K
                    + (0 if direction == "A_to_B" else 1)
                )

                evaluation_seed = (
                    RNG_SEED
                    + 200000 * outer_rep
                    + 1000 * K
                    + (0 if direction == "A_to_B" else 1)
                )

                # ---------------------------------------------------------
                # Rank archetypes using ranking subjects only
                # ---------------------------------------------------------

                ranked, per_arch = rank_archetypes_on_subjects(
                    ranking_subjects,
                    ANALYSIS_TYPE,
                    nfolds=NFOLDS_DECODE,
                    nreps=NREPS_DECODE,
                    seed=ranking_seed,
                )

                per_arch = per_arch.copy()

                per_arch["analysis_type"] = ANALYSIS_TYPE
                per_arch["fit_scope"] = FIT_SCOPE
                per_arch["condition"] = condition
                per_arch["K"] = K
                per_arch["outer_rep"] = outer_rep
                per_arch["direction"] = direction
                per_arch["outer_seed"] = outer_seed
                per_arch["n_ranking_subjects"] = len(
                    ranking_subjects
                )

                outer_rank_parts.append(per_arch)

                # ---------------------------------------------------------
                # Evaluate selected archetypes on held-out subjects
                # ---------------------------------------------------------

                heldout = evaluate_topm_on_subjects(
                    evaluation_subjects,
                    ANALYSIS_TYPE,
                    ranked,
                    top_m_values=TOP_M_VALUES,
                    nfolds=NFOLDS_DECODE,
                    nreps=NREPS_DECODE,
                    seed=evaluation_seed,
                )

                heldout["analysis_type"] = ANALYSIS_TYPE
                heldout["fit_scope"] = FIT_SCOPE
                heldout["condition"] = condition
                heldout["K"] = K
                heldout["outer_rep"] = outer_rep
                heldout["direction"] = direction
                heldout["outer_seed"] = outer_seed

                heldout["n_ranking_subjects"] = len(
                    ranking_subjects
                )

                heldout["n_evaluation_subjects"] = len(
                    evaluation_subjects
                )

                heldout["fit_source_file"] = (
                    fit["fit_source_file"]
                )

                heldout["fit_source_basename"] = (
                    fit["fit_source_basename"]
                )

                outer_run_parts.append(heldout)

            # Only add the outer rep once BOTH directions finish
            run_parts.extend(outer_run_parts)
            rank_parts.extend(outer_rank_parts)

            completed_now = outer_rep + 1

            if completed_now % 5 == 0:
                print(
                    f"  completed outer rep "
                    f"{completed_now}/{N_OUTER_REPS}"
                )

                run_df, rank_df = save_checkpoints(
                    run_parts,
                    rank_parts,
                )

                print("  checkpoint saved")

        # Always save at the end of a condition too
        run_df, rank_df = save_checkpoints(
            run_parts,
            rank_parts,
        )

        print("  condition complete; checkpoint saved")


# -------------------------------------------------------------------------
# Final save
# -------------------------------------------------------------------------

run_df, rank_df = save_checkpoints(
    run_parts,
    rank_parts,
)

print("\nDone.")
print("Held-out run rows:", len(run_df))
print("Ranking rows:", len(rank_df))

In [ ]:

# RUN_CSV = OUTPUT_DIR / "heldout_topm_crossfit_runs.csv"
# RANK_CSV = OUTPUT_DIR / "heldout_per_archetype_ranking_runs.csv"

# run_rows = []
# rank_rows = []

# master_rng = np.random.default_rng(RNG_SEED)

# for K in K_VALUES:

#     print("\n" + "=" * 90)
#     print(f"K = {K}")
#     print("=" * 90)

#     for condition in CONDITIONS_TO_RUN:

#         subjects, fit = get_subjects_for_condition(
#             ANALYSIS_TYPE,
#             FIT_SCOPE,
#             K,
#             condition,
#         )

#         print(
#             f"\n{ANALYSIS_TYPE} | {FIT_SCOPE} | "
#             f"{condition} | K={K} | N={len(subjects)}"
#         )

#         for outer_rep in range(N_OUTER_REPS):

#             outer_seed = int(
#                 master_rng.integers(0, 2**31 - 1)
#             )
#             outer_rng = np.random.default_rng(outer_seed)

#             A_idx, B_idx = split_outer_subject_indices(
#                 len(subjects),
#                 outer_rng
#             )

#             directions = [
#                 ("A_to_B", A_idx, B_idx),
#                 ("B_to_A", B_idx, A_idx),
#             ]

#             for direction, ranking_idx, evaluation_idx in directions:

#                 ranking_subjects = subset_subjects(
#                     subjects,
#                     ranking_idx
#                 )
#                 evaluation_subjects = subset_subjects(
#                     subjects,
#                     evaluation_idx
#                 )

#                 ranking_seed = (
#                     RNG_SEED
#                     + 100000 * outer_rep
#                     + 1000 * K
#                     + (0 if direction == "A_to_B" else 1)
#                 )

#                 evaluation_seed = (
#                     RNG_SEED
#                     + 200000 * outer_rep
#                     + 1000 * K
#                     + (0 if direction == "A_to_B" else 1)
#                 )

#                 ranked, per_arch = rank_archetypes_on_subjects(
#                     ranking_subjects,
#                     ANALYSIS_TYPE,
#                     nfolds=NFOLDS_DECODE,
#                     nreps=NREPS_DECODE,
#                     seed=ranking_seed,
#                 )

#                 per_arch = per_arch.copy()
#                 per_arch["analysis_type"] = ANALYSIS_TYPE
#                 per_arch["fit_scope"] = FIT_SCOPE
#                 per_arch["condition"] = condition
#                 per_arch["K"] = K
#                 per_arch["outer_rep"] = outer_rep
#                 per_arch["direction"] = direction
#                 per_arch["outer_seed"] = outer_seed
#                 per_arch["n_ranking_subjects"] = len(ranking_subjects)
#                 rank_rows.append(per_arch)

#                 heldout = evaluate_topm_on_subjects(
#                     evaluation_subjects,
#                     ANALYSIS_TYPE,
#                     ranked,
#                     top_m_values=TOP_M_VALUES,
#                     nfolds=NFOLDS_DECODE,
#                     nreps=NREPS_DECODE,
#                     seed=evaluation_seed,
#                 )

#                 heldout["analysis_type"] = ANALYSIS_TYPE
#                 heldout["fit_scope"] = FIT_SCOPE
#                 heldout["condition"] = condition
#                 heldout["K"] = K
#                 heldout["outer_rep"] = outer_rep
#                 heldout["direction"] = direction
#                 heldout["outer_seed"] = outer_seed
#                 heldout["n_ranking_subjects"] = len(ranking_subjects)
#                 heldout["n_evaluation_subjects"] = len(evaluation_subjects)
#                 heldout["fit_source_file"] = fit["fit_source_file"]
#                 heldout["fit_source_basename"] = fit["fit_source_basename"]

#                 run_rows.append(heldout)

#             if (outer_rep + 1) % 5 == 0:
#                 print(
#                     f"  completed outer rep "
#                     f"{outer_rep + 1}/{N_OUTER_REPS}"
#                 )

#         run_df = pd.concat(run_rows, ignore_index=True)
#         rank_df = pd.concat(rank_rows, ignore_index=True)

#         run_df.to_csv(RUN_CSV, index=False)
#         rank_df.to_csv(RANK_CSV, index=False)

#         print("  checkpoint saved")

# print("\nDone.")
# print("Held-out run rows:", len(run_df))
# print("Ranking rows:", len(rank_df))


## 6. Summarize held-out top-m results

In [ ]:
# ============================================================
# LOAD SAVED REVISION-04 RESULTS FOR SUMMARY / PLOTTING ONLY
# ============================================================

RUN_CSV = (
    OUTPUT_DIR
    / "heldout_topm_crossfit_runs.csv"
)

RANK_CSV = (
    OUTPUT_DIR
    / "heldout_per_archetype_ranking_runs.csv"
)

assert RUN_CSV.exists(), (
    f"Missing saved run checkpoint: {RUN_CSV.resolve()}"
)

assert RANK_CSV.exists(), (
    f"Missing saved ranking checkpoint: {RANK_CSV.resolve()}"
)

run_df = pd.read_csv(
    RUN_CSV
)

rank_df = pd.read_csv(
    RANK_CSV
)

print(
    "Loaded held-out results:",
    f"{len(run_df):,} rows"
)

print(
    "Loaded ranking results:",
    f"{len(rank_df):,} rows"
)

print("\nSaved K values by condition:")
display(
    run_df.groupby(
        "condition"
    )["K"]
    .apply(
        lambda x: sorted(
            x.astype(int).unique()
        )
    )
)

print("\nSaved outer reps by condition / K:")
display(
    run_df.groupby(
        [
            "condition",
            "K",
        ]
    )["outer_rep"]
    .nunique()
    .unstack(
        fill_value=0
    )
)

In [ ]:

def sem(x):
    x = pd.Series(x).dropna()
    if len(x) <= 1:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))


summary_df = (
    run_df
    .groupby(
        ["analysis_type", "fit_scope", "condition", "K", "top_m"],
        as_index=False
    )
    .agg(
        mean=("mean", "mean"),
        err=("mean", sem),
        std=("mean", "std"),
        count=("mean", "size"),
        mean_full_same_eval=("full_mean_same_eval", "mean"),
        mean_retention=("retention_above_chance", "mean"),
        err_retention=("retention_above_chance", sem),
    )
)

SUMMARY_CSV = OUTPUT_DIR / "heldout_topm_summary.csv"
summary_df.to_csv(SUMMARY_CSV, index=False)

display(summary_df.head(30))
print("Saved:", SUMMARY_CSV)


## 7. Held-out top-m curves

In [ ]:

COND_COLORS = {
    "intact": "purple",
    "word": "green",
    "rest": "black",
}

for top_m in TOP_M_VALUES:

    plt.figure(figsize=(7, 4.5))

    for condition in CONDITIONS_TO_RUN:

        sub = summary_df[
            (summary_df["top_m"] == top_m)
            & (summary_df["condition"] == condition)
        ].sort_values("K")

        if len(sub) == 0:
            continue

        plt.errorbar(
            sub["K"],
            sub["mean"],
            yerr=sub["err"],
            marker="o",
            capsize=4,
            linewidth=2,
            color=COND_COLORS.get(condition, "gray"),
            label=condition
        )

    plt.axhline(
        1 / 300,
        linestyle=":",
        linewidth=1,
        color="gray",
        label="chance"
    )

    plt.xlabel("K")
    plt.ylabel("Held-out timepoint identification accuracy")
    plt.title(
        f"{ANALYSIS_TYPE} AA | {FIT_SCOPE} | "
        f"held-out top-{top_m}"
    )
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


## 8. Sparse retention relative to full-K on the same evaluation participants

In [ ]:

for top_m in TOP_M_VALUES:

    plt.figure(figsize=(7, 4.5))

    for condition in CONDITIONS_TO_RUN:

        sub = summary_df[
            (summary_df["top_m"] == top_m)
            & (summary_df["condition"] == condition)
        ].sort_values("K")

        if len(sub) == 0:
            continue

        plt.errorbar(
            sub["K"],
            sub["mean_retention"],
            yerr=sub["err_retention"],
            marker="o",
            capsize=4,
            linewidth=2,
            color=COND_COLORS.get(condition, "gray"),
            label=condition
        )

    plt.axhline(
        1.0,
        linestyle=":",
        linewidth=1,
        color="gray",
        label="full K"
    )

    plt.xlabel("K")
    plt.ylabel("Fraction of above-chance full-K performance retained")
    plt.title(
        f"{ANALYSIS_TYPE} AA | {FIT_SCOPE} | "
        f"held-out top-{top_m} retention"
    )
    plt.legend(frameon=False)
    plt.tight_layout()
    plt.show()


## 9. Compare with original same-data top-m results

In [ ]:

ORIGINAL_TOPM_CSV = Path(
    f"msaa_condrank_decoding_outputs_{ANALYSIS_TYPE}_{FIT_SCOPE}"
) / "topm_summary.csv"

if ORIGINAL_TOPM_CSV.exists():

    original_df = pd.read_csv(ORIGINAL_TOPM_CSV)

    for top_m in TOP_M_VALUES:
        for condition in CONDITIONS_TO_RUN:

            old = original_df[
                (original_df["condition"].astype(str) == str(condition))
                & (original_df["top_m"].astype(int) == int(top_m))
            ].sort_values("K")

            new = summary_df[
                (summary_df["condition"].astype(str) == str(condition))
                & (summary_df["top_m"].astype(int) == int(top_m))
            ].sort_values("K")

            if len(old) == 0 or len(new) == 0:
                continue

            plt.figure(figsize=(7, 4.5))

            plt.plot(
                old["K"],
                old["mean"],
                marker="o",
                linewidth=2,
                label="Original same-data ranking"
            )

            plt.errorbar(
                new["K"],
                new["mean"],
                yerr=new["err"],
                marker="o",
                capsize=4,
                linewidth=2,
                label="Held-out ranking"
            )

            plt.axhline(
                1 / 300,
                linestyle=":",
                linewidth=1,
                color="gray",
                label="chance"
            )

            plt.xlabel("K")
            plt.ylabel("Timepoint identification accuracy")
            plt.title(
                f"{ANALYSIS_TYPE} AA | {condition} | "
                f"top-{top_m}"
            )
            plt.legend(frameon=False)
            plt.tight_layout()
            plt.show()

else:
    print("Original top-m file not found:", ORIGINAL_TOPM_CSV)


## 10. Archetype selection stability across participant halves

In [ ]:

stability_rows = []

group_cols = [
    "analysis_type",
    "fit_scope",
    "condition",
    "K",
    "outer_rep",
    "direction",
]

for keys, sub in rank_df.groupby(group_cols):

    ranked = (
        sub
        .sort_values(
            ["mean_accuracy", "rank_mean", "archetype"],
            ascending=[False, False, True]
        )
        ["archetype"]
        .astype(int)
        .tolist()
    )

    metadata = dict(zip(group_cols, keys))

    for top_m in TOP_M_VALUES:
        m = min(int(top_m), len(ranked))

        for rank_position, archetype in enumerate(
            ranked[:m],
            start=1
        ):
            stability_rows.append({
                **metadata,
                "top_m": m,
                "rank_position": rank_position,
                "archetype": archetype,
            })

stability_long = pd.DataFrame(stability_rows)

stability_summary = (
    stability_long
    .groupby(
        ["analysis_type", "fit_scope", "condition", "K", "top_m", "archetype"],
        as_index=False
    )
    .agg(
        selected_count=("archetype", "size"),
        mean_rank_position=("rank_position", "mean"),
    )
)

n_crossfits = N_OUTER_REPS * 2
stability_summary["selection_frequency"] = (
    stability_summary["selected_count"] / n_crossfits
)

stability_summary = stability_summary.sort_values(
    ["condition", "K", "top_m", "selection_frequency"],
    ascending=[True, True, True, False]
)

STABILITY_CSV = OUTPUT_DIR / "heldout_archetype_selection_stability.csv"
stability_summary.to_csv(STABILITY_CSV, index=False)

display(
    stability_summary[
        stability_summary["top_m"] == 10
    ].groupby(["condition", "K"], group_keys=False).head(10)
)

print("Saved:", STABILITY_CSV)


In [ ]:
# ============================================================
# DIAGNOSTIC:
# WHAT DOES full_mean_same_eval ACTUALLY LOOK LIKE ACROSS K?
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


SIGNAL_CONDITIONS = ["intact", "word"]


# ------------------------------------------------------------
# Use one top_m row only, because full_mean_same_eval is
# repeated identically across top_m within each evaluation.
# ------------------------------------------------------------

full_diag = run_df[
    (run_df["top_m"] == TOP_M_VALUES[0])
    & (run_df["condition"].isin(SIGNAL_CONDITIONS))
].copy()


# ------------------------------------------------------------
# A->B and B->A belong to the same outer split.
# Average directions within outer_rep first.
# ------------------------------------------------------------

full_by_rep = (
    full_diag
    .groupby(
        ["condition", "K", "outer_rep"],
        as_index=False
    )
    .agg(
        full_accuracy=(
            "full_mean_same_eval",
            "mean"
        )
    )
)


# ------------------------------------------------------------
# Summarize across independent outer splits
# ------------------------------------------------------------

full_summary_diag = (
    full_by_rep
    .groupby(
        ["condition", "K"],
        as_index=False
    )
    .agg(
        mean=("full_accuracy", "mean"),
        std=("full_accuracy", "std"),
        n=("full_accuracy", "size"),
    )
)

full_summary_diag["sem"] = (
    full_summary_diag["std"]
    / np.sqrt(full_summary_diag["n"])
)


display(
    full_summary_diag[
        full_summary_diag["K"].isin(
            [10, 14, 21, 25, 35, 50, 70, 88, 126, 175]
        )
    ]
)


# ------------------------------------------------------------
# Plot exactly the full-K held-out reference
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

for condition in SIGNAL_CONDITIONS:

    sub = (
        full_summary_diag[
            full_summary_diag["condition"] == condition
        ]
        .sort_values("K")
    )

    plt.errorbar(
        sub["K"],
        sub["mean"],
        yerr=sub["sem"],
        marker="o",
        capsize=3,
        linewidth=2,
        label=condition,
    )

plt.axhline(
    1 / 300,
    linestyle=":",
    linewidth=1,
    color="gray",
    label="chance",
)

plt.xlabel("K")
plt.ylabel("Full-K held-out timepoint identification accuracy")
plt.title(
    "Full reconstruction on held-out evaluation participants"
)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OBJECTIVE REPRESENTATIVE K:
# KNEE / ELBOW OF FULL-K HELD-OUT PERFORMANCE
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


SIGNAL_CONDITIONS = ["intact", "word"]


# ------------------------------------------------------------
# 1. Use the direction-averaged full-model summary
#    created in the previous diagnostic cell
# ------------------------------------------------------------

curve = (
    full_summary_diag[
        full_summary_diag["condition"].isin(SIGNAL_CONDITIONS)
    ]
    .pivot(
        index="K",
        columns="condition",
        values="mean",
    )
    .dropna()
    .sort_index()
)


# ------------------------------------------------------------
# 2. Normalize each condition separately to [0, 1]
# ------------------------------------------------------------

norm = curve.copy()

for condition in SIGNAL_CONDITIONS:

    y = curve[condition].values.astype(float)

    norm[condition] = (
        (y - y.min())
        / (y.max() - y.min())
    )


# Average normalized intact + word
norm["mean_signal"] = (
    norm[SIGNAL_CONDITIONS]
    .mean(axis=1)
)


# ------------------------------------------------------------
# 3. Normalize K itself to [0, 1]
# ------------------------------------------------------------

x = norm.index.values.astype(float)
y = norm["mean_signal"].values.astype(float)

x_norm = (
    (x - x.min())
    / (x.max() - x.min())
)


# ------------------------------------------------------------
# 4. Distance from straight line joining first and last points
#
# For a concave increasing curve, the knee is the point
# furthest ABOVE the endpoint-to-endpoint line.
# ------------------------------------------------------------

line = (
    y[0]
    + (y[-1] - y[0]) * x_norm
)

distance = y - line

norm["knee_distance"] = distance


REPRESENTATIVE_K = int(
    norm["knee_distance"].idxmax()
)


print(
    "OBJECTIVE KNEE:",
    REPRESENTATIVE_K
)

display(norm)


# ------------------------------------------------------------
# 5. Plot
# ------------------------------------------------------------

plt.figure(figsize=(8, 5))

plt.plot(
    norm.index,
    norm["mean_signal"],
    marker="o",
    linewidth=2,
    label="Mean normalized intact + word",
)

plt.plot(
    norm.index,
    line,
    linestyle="--",
    linewidth=1,
    label="Endpoint reference line",
)

plt.axvline(
    REPRESENTATIVE_K,
    linestyle=":",
    linewidth=2,
    label=f"Knee: K={REPRESENTATIVE_K}",
)

plt.xlabel("K")
plt.ylabel("Normalized held-out full-K performance")
plt.title(
    "Objective knee in held-out full-reconstruction performance"
)
plt.legend(frameon=False)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# OBJECTIVE REPRESENTATIVE K FROM HELD-OUT TOP-10 PERFORMANCE
# ============================================================

TOP_M_INTERPRET = 3
SIGNAL_CONDITIONS = ["intact", "word"]


# ------------------------------------------------------------
# 1. Held-out top-10 absolute accuracy
# ------------------------------------------------------------

top10 = summary_df[
    (summary_df["top_m"] == TOP_M_INTERPRET)
    & (summary_df["condition"].isin(SIGNAL_CONDITIONS))
].copy()


perf = (
    top10
    .pivot(
        index="K",
        columns="condition",
        values="mean",
    )
    .dropna(subset=SIGNAL_CONDITIONS)
    .sort_index()
)


# ------------------------------------------------------------
# 2. Normalize each condition to its own best observed value
# ------------------------------------------------------------

for condition in SIGNAL_CONDITIONS:

    perf[f"{condition}_relative"] = (
        perf[condition]
        / perf[condition].max()
    )


# ------------------------------------------------------------
# 3. Joint score
#
# Conservative: how well does this K do in its WORSE
# stimulus-driven condition?
# ------------------------------------------------------------

perf["joint_score"] = (
    perf[
        [
            "intact_relative",
            "word_relative",
        ]
    ]
    .min(axis=1)
)


# ------------------------------------------------------------
# 4. Choose K with best joint held-out top-10 performance
# ------------------------------------------------------------

REPRESENTATIVE_K = int(
    perf["joint_score"].idxmax()
)


print("Held-out top-10 K comparison:")
display(
    perf.sort_values(
        "joint_score",
        ascending=False,
    )
)

print(
    "\nOBJECTIVE REPRESENTATIVE K:",
    REPRESENTATIVE_K,
)

In [ ]:
# ============================================================
# SELECT REPRODUCIBLY INFORMATIVE ARCHETYPES
# ============================================================

N_ARCHETYPES_TO_SHOW = 5

stab = stability_summary[
    (stability_summary["K"] == REPRESENTATIVE_K)
    & (stability_summary["top_m"] == TOP_M_INTERPRET)
].copy()

# Selection frequencies by condition
freq = (
    stab
    .pivot_table(
        index="archetype",
        columns="condition",
        values="selection_frequency",
        aggfunc="mean",
        fill_value=0,
    )
)

# Make sure all conditions exist as columns
for condition in ["intact", "word", "rest"]:
    if condition not in freq.columns:
        freq[condition] = 0.0

# Primary criterion:
# must generalize across BOTH stimulus-driven conditions
freq["shared_signal_frequency"] = (
    freq[["intact", "word"]].min(axis=1)
)

# Secondary criterion:
# overall frequency across intact + word
freq["mean_signal_frequency"] = (
    freq[["intact", "word"]].mean(axis=1)
)

selected = (
    freq
    .sort_values(
        [
            "shared_signal_frequency",
            "mean_signal_frequency",
        ],
        ascending=False,
    )
    .head(N_ARCHETYPES_TO_SHOW)
)

SELECTED_ARCHETYPES = (
    selected.index
    .astype(int)
    .tolist()
)

print(
    "Representative K:",
    REPRESENTATIVE_K
)

print(
    "\nObjectively selected archetypes:",
    SELECTED_ARCHETYPES
)

display(selected)

In [ ]:
# ============================================================
# LOCK REPRESENTATIVE MODEL AND SELECT FIVE ARCHETYPES
# ============================================================

REPRESENTATIVE_K = 46
TOP_M_INTERPRET = 5
N_ARCHETYPES_TO_SHOW = 5


# ------------------------------------------------------------
# Selection stability at K=46
# ------------------------------------------------------------

stab = stability_summary[
    (stability_summary["K"] == REPRESENTATIVE_K)
    & (stability_summary["top_m"] == TOP_M_INTERPRET)
].copy()


# Selection frequency by condition
freq = (
    stab
    .pivot_table(
        index="archetype",
        columns="condition",
        values="selection_frequency",
        aggfunc="mean",
        fill_value=0,
    )
)

for condition in ["intact", "word", "rest"]:
    if condition not in freq.columns:
        freq[condition] = 0.0


# ------------------------------------------------------------
# Require reproducibility in BOTH stimulus-driven conditions
# ------------------------------------------------------------

freq["shared_signal_frequency"] = (
    freq[["intact", "word"]].min(axis=1)
)

freq["mean_signal_frequency"] = (
    freq[["intact", "word"]].mean(axis=1)
)


# ------------------------------------------------------------
# Select five most reproducibly informative archetypes
# ------------------------------------------------------------

selected = (
    freq
    .sort_values(
        [
            "shared_signal_frequency",
            "mean_signal_frequency",
        ],
        ascending=False,
    )
    .head(N_ARCHETYPES_TO_SHOW)
)

SELECTED_ARCHETYPES = (
    selected.index
    .astype(int)
    .tolist()
)

print("Representative K:", REPRESENTATIVE_K)
print("Primary sparse subset: top-", TOP_M_INTERPRET, sep="")
print("\nSelected archetypes:", SELECTED_ARCHETYPES)

display(selected)


## Interpretation for the revision

If the sparse pattern survives, the rebuttal can say that archetypes were ranked on one participant subset and evaluated on a non-overlapping held-out subset, with the roles reversed across repeated random splits.

If held-out performance drops, that drop should be reported and the sparse claim narrowed accordingly. The held-out estimates should replace the original same-data top-m estimates in the revised manuscript.
